# 02 — SVI Segmentation (multi-city)

Runs Mask2Former panoptic segmentation (Mapillary Vistas v1.2) over every
reconciled point's front-view image, for every city and both classes --
`01`'s `reconciled_points.parquet` now covers all cities in
`configs/paths.yaml`'s `cities` list combined, with city-prefixed
`point_id`s (e.g. `bog_positive_123`), so this notebook's existing
per-point loop needs no city-specific logic at all: `image_path` was
already resolved to each point's correct per-city front-view file back
in `01`, and every output here is keyed by the already-globally-unique
`point_id`. Saves the raw segmentation map + per-segment info per point
-- no overlay/panel images here, those belong to 03's graph-aware
visualization.

Checkpointed per point (skip if output already exists) and logged via
`src/manifest.py`. **GPU runtime required.**

Uses `src/segmentation.py` and `src/manifest.py` — real modules, not stubs.

In [ ]:
# ── Clone/update repo, mount Drive ──────────────────────────────────────
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# torch is expected pre-installed on Colab's GPU runtime — not reinstalled here.
!pip install -q transformers accelerate pillow tqdm pyyaml pandas seaborn matplotlib

In [ ]:
import yaml
from pathlib import Path

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/svg_schema.yaml") as f:
    svg_cfg = yaml.safe_load(f)

INTERIM_DIR = Path(paths_cfg["interim_dir"])
SEG_OUT_DIR = INTERIM_DIR / "segmentation"
LOG_PATH = INTERIM_DIR / "segmentation_log.csv"
SEG_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Checkpoint: {svg_cfg['mapillary_checkpoint']}")
print(f"Mask confidence threshold: {svg_cfg['mask_confidence_threshold']}")
print(f"Output dir: {SEG_OUT_DIR}")

In [ ]:
import manifest
import segmentation as seg

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("⚠️  No GPU detected — this will be very slow. Switch runtime type to GPU.")

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────
processor, model, device, class_names, stuff_ids, id2label = seg.load_model(
    svg_cfg["mapillary_checkpoint"]
)
print(f"Device: {device}")
print(f"Num classes: {len(class_names)}")
print(f"Fusing {len(stuff_ids)} stuff classes: "
      f"{sorted(class_names[i] for i in stuff_ids)}")

In [ ]:
# ── Cross-check the checkpoint's actual stuff/thing split against our
#    schema's expectations — catches a silent mismatch before it corrupts
#    every downstream graph, rather than discovering it in 03. ────────────
problems = seg.validate_against_schema(
    class_names, stuff_ids,
    svg_cfg["tier1_thing_classes"],
    svg_cfg["stuff_classes_for_island_labeling"],
)

if problems:
    print("❌ Schema mismatch detected:")
    for p in problems:
        print(f"   - {p}")
    raise RuntimeError("Fix svg_schema.yaml or investigate the checkpoint before proceeding.")
else:
    print("✅ Checkpoint's stuff/thing split matches svg_schema.yaml exactly.")

In [ ]:
# ── Load reconciled points from 01 ──────────────────────────────────────
import pandas as pd

reconciled = pd.read_parquet(INTERIM_DIR / "reconciled_points.parquet")
all_point_ids = reconciled["point_id"].tolist()
print(f"Total reconciled points: {len(all_point_ids)}")

pending = manifest.pending_items(all_point_ids, SEG_OUT_DIR)
print(f"Already done: {len(all_point_ids) - len(pending)}  |  Pending: {len(pending)}")

In [ ]:
# ── Main inference loop — checkpointed, resumable ───────────────────────
from PIL import Image
from tqdm.auto import tqdm

path_lookup = dict(zip(reconciled["point_id"], reconciled["image_path"]))

for point_id in tqdm(pending, desc="Segmenting"):
    try:
        image = Image.open(path_lookup[point_id]).convert("RGB")
        seg_map, segments_info = seg.infer_panoptic(
            image, processor, model, device, stuff_ids,
            confidence_threshold=svg_cfg["mask_confidence_threshold"],
        )
        seg.save_result(SEG_OUT_DIR, point_id, seg_map, segments_info)
        manifest.append_log(LOG_PATH, point_id, "segmentation", "ok")

    except Exception as e:
        manifest.append_log(LOG_PATH, point_id, "segmentation", "error", str(e))
        tqdm.write(f"  ⚠️  {point_id}: {e}")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────
log = manifest.load_log(LOG_PATH)
n_done = len(manifest.pending_items(all_point_ids, SEG_OUT_DIR))
print(f"Remaining pending after this run: {n_done} / {len(all_point_ids)}")

if "status" in log.columns and (log["status"] == "error").any():
    errors = log[log["status"] == "error"]
    print(f"\n⚠️  {len(errors)} points failed — re-running this notebook will retry them")
    print(f"    (they have no output file yet, so they're still 'pending').")
    display(errors[["point_id", "error", "timestamp"]].tail(20))
else:
    print("\n✅ No errors logged.")

In [ ]:
# ── Light QC: eyeball a couple of results (not the graph-aware overlay — ──
#    that's 03's job) ───────────────────────────────────────────────────
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

sns.set_style("white")
palette = seg.get_palette(class_names)  # official Mapillary colors when they match

sample_ids = [pid for pid in all_point_ids if manifest.is_done(SEG_OUT_DIR, pid)][:2]

for point_id in sample_ids:
    seg_map, segments_info = seg.load_result(SEG_OUT_DIR, point_id)
    image = Image.open(path_lookup[point_id]).convert("RGB")

    seg_rgb = np.zeros((*seg_map.shape, 3), dtype=np.uint8)
    for s in segments_info:
        seg_rgb[seg_map == s["id"]] = palette[s["label_id"]]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(image); axes[0].axis("off"); axes[0].set_title(point_id, fontsize=10)
    axes[1].imshow(seg_rgb); axes[1].axis("off"); axes[1].set_title(f"{len(segments_info)} segments", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
print("Next: 03_svg_construction.ipynb")